In [9]:
!pip install -q transformers sentence-transformers datasets ipywidgets


In [10]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import ipywidgets as widgets
from IPython.display import display


In [11]:
from datasets import load_dataset


In [12]:
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train"
)


In [13]:
df = pd.DataFrame(dataset)

# Reduce size for fast demo (recommended)
df = df.sample(3000, random_state=42).reset_index(drop=True)

df.head()

,flags,instruction,category,intent,response
0,BLZ,I can't talk with a human agent,CONTACT,contact_human_agent,Thank you for reaching out! I'm sorry to hear ...
1,BLMZ,I have got to locate hte bills from {{Person N...,INVOICE,check_invoice,I'll get right on it! I understand your need t...
2,BCELM,"I cannot pay, help me to inform of a problem w...",PAYMENT,payment_issue,I apologize for the inconvenience caused by th...
3,BL,I want help speaking to customer service,CONTACT,contact_customer_service,Grateful for your contact! I get the sense tha...
4,BLZ,I try to see th accepted payment options,PAYMENT,check_payment_methods,Thank you for reaching out! I completely under...


In [6]:
type(df.loc[0, "response"]), df.loc[0, "response"]


(str,
 "Thank you for reaching out! I'm sorry to hear that you're facing difficulties in connecting with a human agent. Please rest assured that I'm here to assist you and resolve this issue. Could you please provide me with some more details about the problem you're experiencing? This will help me find the best solution for you.")

In [14]:
def clean_response(resp):
    return resp[1] if isinstance(resp, tuple) else resp

df["clean_response"] = df["response"].apply(clean_response)

# ✅ Verify it is now string, not tuple
print(type(df.loc[0, "clean_response"]))
print(df.loc[0, "clean_response"][:200])


<class 'str'>
Thank you for reaching out! I'm sorry to hear that you're facing difficulties in connecting with a human agent. Please rest assured that I'm here to assist you and resolve this issue. Could you please


In [15]:
import pandas as pd


In [16]:
embed_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
questions = df["instruction"].tolist()

question_embeddings = embed_model.encode(
    questions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

question_embeddings = question_embeddings / np.linalg.norm(question_embeddings, axis=1, keepdims=True)


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

In [18]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # IMPORTANT FIX

model = AutoModelForCausalLM.from_pretrained(model_name)
model.to("cuda" if torch.cuda.is_available() else "cpu")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [19]:
def generate_answer(query):
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    scores = (question_embeddings @ query_embedding.T).squeeze()
    best_idx = int(scores.argmax())

    source = df.loc[best_idx, "clean_response"]  # ✅ ONLY this column

    prompt = f"""
Answer the customer question using the information below.

Question:
{query}

Information:
{source}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=120)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("Answer:")[-1].strip()

    return answer, source


In [21]:
ans, src = generate_answer("My account is locked")
print("ANS TYPE:", type(ans))
print("SRC TYPE:", type(src))
print("\nAI:", ans)
print("\nSOURCE:", src[:200])


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


ANS TYPE: <class 'str'>
SRC TYPE: <class 'str'>

AI: Thank you for your time and understanding.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience.

Thank you for your patience

SOURCE: Sure! I completely understand the significance of recovering your account password and will gladly assist you throughout the process.

To retrieve your account password, follow these steps:

1. Go to 


In [23]:
input_box = widgets.Text(
    placeholder="Ask a customer support question...",
    description="You:",
    disabled=False
)

output_box = widgets.Output()

def on_submit(change):
    user_query = input_box.value.strip()
    if not user_query:
        return
    with output_box:
        print(f"You: {user_query}")
        answer, source = generate_answer(user_query)
        print(f"AI: {answer}")
        print("\n📌 Source used:")
        print(source)
        print()
    input_box.value = ""

input_box.on_submit(on_submit)
display(input_box, output_box)


Text(value='', description='You:', placeholder='Ask a customer support question...')

Output()

In [22]:
dim = question_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(question_embeddings)
print("FAISS index is ready!")

FAISS index is ready!


In [23]:
def retrieve(query, k=3):
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_vec, k)
    return [df.iloc[i] for i in I[0]]

In [39]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [52]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to("cuda" if torch.cuda.is_available() else "cpu")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [74]:
def clean_response(resp):
    if isinstance(resp, tuple):
        return resp[1]   # take full response text
    return resp

df["clean_response"] = df["response"].apply(clean_response)


In [75]:
def generate_answer(query):
    # Embed query
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Similarity search
    scores = question_embeddings @ query_embedding.T
    best_idx = scores.argmax()

    source = df.iloc[best_idx]["clean_response"]

    prompt = f"""
Answer the customer question using the information below.

Question:
{query}

Information:
{source}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("Answer:")[-1].strip()

    return answer, source


In [76]:
def on_submit(change):
    user_query = input_box.value.strip()
    if not user_query:
        return

    with output_box:
        print(f"You: {user_query}")
        answer, source = generate_answer(user_query)
        print(f"AI: {answer}")
        print("\n📌 Source used:")
        print(source)
        print()

    input_box.value = ""


In [77]:
input_box.on_submit(on_submit)
display(input_box, output_box)


Text(value='', description='You:', placeholder='Ask a customer support question...')

Output()

In [59]:
import ipywidgets as widgets
from IPython.display import display

In [60]:
def on_submit(change):
    user_query = input_box.value.strip()
    if not user_query:
        return

    with output_box:
        print(f"You: {user_query}")
        answer = generate_answer(user_query)
        print(f"AI: {answer}\n")

    input_box.value = ""


In [61]:
input_box.on_submit(on_submit)
display(input_box, output_box)


Text(value='', description='You:', placeholder='Ask a customer support question...')

Output()